In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

## Load 2025 NCAA Outdoor Championship Finals Results

In [ ]:
df = pd.read_csv('results_2025.csv')

HURDLE_EVENTS   = {'110MH', '100MH', '400MH'}
SPRINT_EVENTS   = {'100M', '200M', '400M', '4x100M', '4x400M'}
MID_EVENTS      = {'800M', '1500M'}
DISTANCE_EVENTS = {'3000SC', '5000M', '10000M'}
JUMP_EVENTS     = {'High Jump', 'Pole Vault', 'Long Jump', 'Triple Jump'}
THROW_EVENTS    = {'Shot Put', 'Discus', 'Hammer', 'Javelin'}
MAIN_CATS       = ['Sprints', 'Hurdles', 'Mid-Distance', 'Distance', 'Jumps', 'Throws']

def event_category(event):
    if event in HURDLE_EVENTS:   return 'Hurdles'
    if event in SPRINT_EVENTS:   return 'Sprints'
    if event in MID_EVENTS:      return 'Mid-Distance'
    if event in DISTANCE_EVENTS: return 'Distance'
    if event in JUMP_EVENTS:     return 'Jumps'
    if event in THROW_EVENTS:    return 'Throws'
    return 'Multi'

df['cat_group'] = df['event'].apply(event_category)

df_m = df[df['gender'] == 'M'].copy()
df_w = df[df['gender'] == 'W'].copy()

print(f"Men's rows: {len(df_m)}  |  Women's rows: {len(df_w)}")

In [ ]:
def compute_standard(df_g):
    std = (
        df_g.groupby('school')['points']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'points': 'pts_standard'})
    )
    std['rank_standard'] = std['pts_standard'].rank(ascending=False, method='min').astype(int)
    return std


def compute_event_cap(df_g, cap):
    """Team standings under an n-event individual cap (relay points unchanged)."""
    indiv_scored = df_g[(~df_g['is_relay']) & (df_g['points'] > 0)].copy()
    athlete_capped = (
        indiv_scored.groupby(['name', 'school'])['points']
        .apply(lambda pts: pts.nlargest(cap).sum())
        .reset_index(name='pts_indiv_capped')
    )
    team_indiv = athlete_capped.groupby('school')['pts_indiv_capped'].sum().reset_index()
    relay_pts = (
        df_g[df_g['is_relay']]
        .groupby('school')['points']
        .sum()
        .reset_index()
        .rename(columns={'points': 'pts_relay'})
    )
    result = team_indiv.merge(relay_pts, on='school', how='outer').fillna(0)
    col = f'pts_{cap}event'
    result[col] = result['pts_indiv_capped'] + result['pts_relay']
    result[f'rank_{cap}event'] = result[col].rank(ascending=False, method='min').astype(int)
    return result.drop(columns='pts_indiv_capped')


def get_multi_event_athletes(df_g):
    """Athletes scoring in 2+ individual events with points lost under each cap."""
    indiv_scored = df_g[(~df_g['is_relay']) & (df_g['points'] > 0)].copy()
    ath = (
        indiv_scored.groupby(['name', 'school'])
        .apply(lambda g: pd.Series({
            'n_events':  len(g),
            'events':    list(g['event']),
            'pts_each':  list(g['points']),
            'total_pts': g['points'].sum(),
        }), include_groups=False)
        .reset_index()
    )
    ath['pts_cap1'] = ath['pts_each'].apply(lambda p: sum(sorted(p, reverse=True)[:1]))
    ath['pts_cap2'] = ath['pts_each'].apply(lambda p: sum(sorted(p, reverse=True)[:2]))
    ath['lost_1event'] = ath['total_pts'] - ath['pts_cap1']
    ath['lost_2event'] = ath['total_pts'] - ath['pts_cap2']
    return ath[ath['n_events'] >= 2].sort_values('total_pts', ascending=False)


def compute_equal_cat(df_g):
    """Equal category share across Sprints, Hurdles, Mid-Distance, Distance, Jumps, Throws."""
    df_main = df_g[(df_g['cat_group'].isin(MAIN_CATS)) & (df_g['points'] > 0)]
    cat_pool = df_main.groupby('cat_group')['points'].sum().rename('cat_pool')
    team_cat_pts = (
        df_main.groupby(['school', 'cat_group'])['points']
        .sum()
        .reset_index()
        .rename(columns={'points': 'pts_cat'})
        .merge(cat_pool.reset_index(), on='cat_group')
    )
    team_cat_pts['cat_share'] = team_cat_pts['pts_cat'] / team_cat_pts['cat_pool']
    equal_share_value = cat_pool.sum() / len(MAIN_CATS)
    team_cat_pts['pts_equal'] = team_cat_pts['cat_share'] * equal_share_value
    alt = (
        team_cat_pts.groupby('school')['pts_equal']
        .sum()
        .reset_index()
        .rename(columns={'pts_equal': 'pts_equal_cat'})
        .sort_values('pts_equal_cat', ascending=False)
        .reset_index(drop=True)
    )
    alt['rank_equal_cat'] = alt['pts_equal_cat'].rank(ascending=False, method='min').astype(int)
    return alt, team_cat_pts


def compute_all_athletes(df_g):
    """Score all finishers across all rounds.

    Finalists score from the final; prelim-only athletes (didn't make the final)
    are ranked by their prelim place and offset after all finalists.
    """
    finals  = df_g[df_g['is_final']  & df_g['place'].notna()].copy()
    prelims = df_g[~df_g['is_final'] & df_g['place'].notna()].copy()

    # Remove athletes from prelim data who advanced to the final
    if not prelims.empty:
        final_names = finals.groupby('event')['name'].apply(set)
        mask = prelims.apply(
            lambda r: r['name'] not in final_names.get(r['event'], set()), axis=1
        )
        prelims = prelims[mask].copy()

    # Count finalists per event (used for the place offset)
    n_finalists = finals.groupby('event')['place'].count().rename('n_finalists')

    if not prelims.empty:
        prelims = prelims.merge(n_finalists.reset_index(), on='event', how='left')
        prelims['n_finalists'] = prelims['n_finalists'].fillna(0).astype(int)
        # Rank prelim-only athletes within each event by their prelim place
        prelims['prelim_rank'] = (
            prelims.groupby('event')['place']
            .rank(method='first', ascending=True)
            .astype(int)
        )
        prelims['effective_place'] = prelims['n_finalists'] + prelims['prelim_rank']

    # Total athletes per event = finalists + prelim-only
    n_prelim_only = (
        prelims.groupby('event').size().rename('n_prelim_only')
        if not prelims.empty
        else pd.Series(name='n_prelim_only', dtype=int)
    )
    totals = (
        n_finalists.reset_index()
        .merge(n_prelim_only.reset_index(), on='event', how='outer')
        .fillna(0)
    )
    totals['n_total'] = totals['n_finalists'] + totals['n_prelim_only']

    finals = finals.merge(totals[['event', 'n_total']], on='event')
    finals['pts_all'] = (finals['n_total'] + 1 - finals['place'].astype(int)).clip(lower=0)

    if not prelims.empty:
        prelims = prelims.merge(totals[['event', 'n_total']], on='event', how='left')
        prelims['pts_all'] = (prelims['n_total'] + 1 - prelims['effective_place']).clip(lower=0)
        scored = pd.concat([finals[['school', 'pts_all']], prelims[['school', 'pts_all']]])
    else:
        scored = finals[['school', 'pts_all']]

    alt = (
        scored.groupby('school')['pts_all']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'pts_all': 'pts_all_athletes'})
    )
    alt['rank_all_athletes'] = alt['pts_all_athletes'].rank(ascending=False, method='min').astype(int)
    return alt


def compute_all_scores(df_g):
    standard         = compute_standard(df_g)
    alt_1ev          = compute_event_cap(df_g, 1)
    alt_2ev          = compute_event_cap(df_g, 2)
    alt_eq, team_cat = compute_equal_cat(df_g)
    alt_all          = compute_all_athletes(df_g)
    multi_ath        = get_multi_event_athletes(df_g)
    relay_pts = (
        df_g[df_g['is_relay']].groupby('school')['points']
        .sum().reset_index().rename(columns={'points': 'pts_relay'})
    )

    comparison = (
        standard
        .merge(alt_1ev[['school', 'pts_1event', 'rank_1event']], on='school', how='outer')
        .merge(alt_2ev[['school', 'pts_2event', 'rank_2event']], on='school', how='outer')
        .merge(alt_eq[['school', 'pts_equal_cat', 'rank_equal_cat']], on='school', how='left')
        .merge(alt_all[['school', 'pts_all_athletes', 'rank_all_athletes']], on='school', how='outer')
        .sort_values('rank_standard')
    )
    for col, rank_col in [('pts_1event', 'rank_1event'), ('pts_2event', 'rank_2event'),
                          ('pts_all_athletes', 'rank_all_athletes')]:
        comparison[col]      = comparison[col].fillna(0)
        comparison[rank_col] = comparison[rank_col].fillna(0).astype(int)
    comparison['d_equal_cat']    = (comparison['rank_standard'] - comparison['rank_equal_cat']).where(comparison['rank_equal_cat'].notna())
    comparison['d_all_athletes'] = comparison['rank_standard'] - comparison['rank_all_athletes']
    comparison['d_1event']       = comparison['rank_standard'] - comparison['rank_1event']
    comparison['d_2event']       = comparison['rank_standard'] - comparison['rank_2event']

    return standard, alt_1ev, alt_2ev, alt_eq, alt_all, comparison, multi_ath, team_cat, relay_pts


std_m, alt1ev_m, alt2ev_m, alt_eq_m, alt_all_m, comp_m, multi_m, cat_m, rel_m = compute_all_scores(df_m)
std_w, alt1ev_w, alt2ev_w, alt_eq_w, alt_all_w, comp_w, multi_w, cat_w, rel_w = compute_all_scores(df_w)
print("Scores computed for both genders.")

---
## 1. Standard NCAA Scoring
Individual: **10-8-6-5-4-3-2-1** for places 1–8 | Relay: **20-16-12-10-8-6-4-2**

In [ ]:
print("MEN — Top 15")
display(std_m.head(15))

print("WOMEN — Top 15")
display(std_w.head(15))

In [ ]:
# Category breakdown for top 10 each gender
def cat_breakdown(df_g, std, label):
    top = std.head(10)['school'].tolist()
    bd = (
        df_g[df_g['school'].isin(top)]
        .groupby(['school','cat_group'])['points']
        .sum()
        .unstack(fill_value=0)
        .loc[top]
    )
    bd['Total'] = bd.sum(axis=1)
    print(f"{label} — points by category (top 10 schools)")
    display(bd.sort_values('Total', ascending=False))

cat_breakdown(df_m, std_m, 'MEN')
cat_breakdown(df_w, std_w, 'WOMEN')

---
## 2. Alternative Scoring 1 & 2: Event Caps per Athlete

**Alt 1 — 1-Event Cap**: each athlete may score in only their **single best individual event**.  
**Alt 2 — 2-Event Cap**: each athlete may score in at most **2 individual events**.

Relay points are unchanged in both (relay rosters aren't in published results, so a relay-inclusive cap can't be enforced directly). The table below shows multi-event individual scorers and the points each would lose under each cap.

In [ ]:
cols = ['name', 'school', 'n_events', 'events', 'pts_each', 'total_pts', 'lost_1event', 'lost_2event']

print("MEN — athletes scoring in 2+ individual events:")
display(multi_m[cols])

print("\nWOMEN — athletes scoring in 2+ individual events:")
display(multi_w[cols])

In [ ]:
print("MEN — top 15 under 1-event cap:")
display(alt1ev_m.sort_values('rank_1event').head(15))

print("MEN — top 15 under 2-event cap:")
display(alt2ev_m.sort_values('rank_2event').head(15))

print("WOMEN — top 15 under 1-event cap:")
display(alt1ev_w.sort_values('rank_1event').head(15))

print("WOMEN — top 15 under 2-event cap:")
display(alt2ev_w.sort_values('rank_2event').head(15))

In [ ]:
def relay_dep_table(alt, std, label):
    t = (
        alt.merge(std, on='school')
        [['school', 'pts_relay', 'pts_standard']]
        .assign(relay_pct=lambda x: x['pts_relay'] / x['pts_standard'] * 100)
        .query('pts_relay > 0')
        .sort_values('relay_pct', ascending=False)
        .head(12)
    )
    print(f"{label} — relay dependence (most exposed under strict event caps):")
    display(t)

relay_dep_table(alt2ev_m, std_m, 'MEN')
relay_dep_table(alt2ev_w, std_w, 'WOMEN')

---
## 3. Alternative Scoring 3: Equal Category Share

Force **Sprints, Hurdles, Mid-Distance, Distance, Jumps, Throws** to each contribute equally.  
Each category's pool is computed per gender separately — so the normalization reflects how that gender's competition is distributed.

In [ ]:
def show_cat_pool(df_g, label):
    pool = (
        df_g[(df_g['cat_group'].isin(MAIN_CATS)) & (df_g['points'] > 0)]
        .groupby('cat_group')['points']
        .sum()
        .sort_values(ascending=False)
    )
    print(f"{label} — scoring points per category:")
    display(pool)

show_cat_pool(df_m, 'MEN')
show_cat_pool(df_w, 'WOMEN')

In [ ]:
print("MEN — top 15 under equal category share (6 categories incl. Hurdles):")
display(alt_eq_m.head(15))

print("WOMEN — top 15 under equal category share:")
display(alt_eq_w.head(15))

---
## 4. Alternative Scoring 4: All Athletes Score

Every finisher earns points. **Last place = 1 pt**, incrementing by 1 up to 1st place.  
If an event has *n* finishers, 1st earns *n* pts, 2nd earns *n − 1*, …, last earns 1.  
DNFs and DQs with no recorded place receive 0.

In [ ]:
print("MEN — top 15 under all-athletes scoring:")
display(alt_all_m.head(15))

print("WOMEN — top 15 under all-athletes scoring:")
display(alt_all_w.head(15))

---
## 5. Comparison: All Four Alternatives

In [ ]:
cols_show = ['school', 'pts_standard', 'rank_standard',
             'rank_1event', 'd_1event',
             'rank_2event', 'd_2event',
             'pts_equal_cat', 'rank_equal_cat', 'd_equal_cat',
             'rank_all_athletes', 'd_all_athletes']

print("MEN — top 20:")
display(comp_m[cols_show].head(20))

print("WOMEN — top 20:")
display(comp_w[cols_show].head(20))

In [ ]:
def show_movers(comp, label, n=8):
    def movers_for(sub, rank_col, d_col, alt_label):
        print(f"  {alt_label} — biggest risers:")
        display(sub.sort_values(d_col, ascending=False)[['school', 'rank_standard', rank_col, d_col]].head(n))
        print(f"  {alt_label} — biggest fallers:")
        display(sub.sort_values(d_col, ascending=True)[['school', 'rank_standard', rank_col, d_col]].head(n))

    print(f"\n{label}")
    movers_for(comp[comp['rank_1event'] > 0],    'rank_1event',       'd_1event',       'Alt 1 (1-event cap)')
    movers_for(comp[comp['rank_2event'] > 0],    'rank_2event',       'd_2event',       'Alt 2 (2-event cap)')
    movers_for(comp.dropna(subset=['d_equal_cat']), 'rank_equal_cat', 'd_equal_cat',    'Alt 3 (equal category)')
    movers_for(comp[comp['rank_all_athletes'] > 0], 'rank_all_athletes', 'd_all_athletes', 'Alt 4 (all athletes)')

show_movers(comp_m, 'MEN')
show_movers(comp_w, 'WOMEN')

---
## 6. Visualizations

In [ ]:
ordered_cats = ['Sprints', 'Hurdles', 'Mid-Distance', 'Distance', 'Jumps', 'Throws', 'Multi']

def make_cat_data(df_g, std, n=12):
    top = std.head(n)['school'].tolist()
    cd = (
        df_g[df_g['school'].isin(top)]
        .groupby(['school', 'cat_group'])['points']
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=ordered_cats, fill_value=0)
    )
    return cd.loc[cd.sum(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(2, 1, figsize=(13, 10))
cmap = plt.cm.Set2
colors = [cmap(i / 7) for i in range(7)]

for ax, (df_g, std), label in zip(
    axes,
    [(df_m, std_m), (df_w, std_w)],
    ["Men's", "Women's"]
):
    cd = make_cat_data(df_g, std)
    bottom = np.zeros(len(cd))
    for i, cat in enumerate(ordered_cats):
        if cat in cd.columns:
            ax.bar(cd.index, cd[cat], bottom=bottom, color=colors[i], label=cat, alpha=0.9)
            bottom += cd[cat].values
    ax.set_title(f"{label} — Standard Points by Category (top 12)", fontweight='bold')
    ax.set_ylabel('Points')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Category', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

plt.suptitle('2025 NCAA Outdoor T&F: Category Breakdown', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('category_breakdown.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Slope charts: standard vs equal-cat rank — men left, women right
def slope_chart(ax, comp, title, top_n=20):
    sub = comp[comp['rank_standard'] <= top_n].copy()
    for _, row in sub.iterrows():
        r_std = row['rank_standard']
        r_eq  = row['rank_equal_cat']
        color = ('#2ca02c' if r_eq < r_std else
                 '#d62728' if r_eq > r_std else '#aaaaaa')
        ax.plot([0, 1], [r_std, r_eq], color=color, alpha=0.75, linewidth=2)
        ax.text(-0.04, r_std, row['school'], ha='right', va='center', fontsize=7.5)
        ax.text(1.04,  r_eq,  row['school'], ha='left',  va='center', fontsize=7.5)
    ax.set_xlim(-0.65, 1.65)
    ax.invert_yaxis()
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Standard', 'Equal Cat.'], fontsize=9)
    ax.set_ylabel('Rank (1 = best)')
    ax.set_title(title, fontweight='bold')
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    green_p = mpatches.Patch(color='#2ca02c', label='Improves')
    red_p   = mpatches.Patch(color='#d62728', label='Drops')
    gray_p  = mpatches.Patch(color='#aaaaaa', label='No change')
    ax.legend(handles=[green_p, red_p, gray_p], loc='lower right', fontsize=7)

fig, (ax_m, ax_w) = plt.subplots(1, 2, figsize=(16, 10))
slope_chart(ax_m, comp_m, "Men's: Standard vs Equal Category (top 20)")
slope_chart(ax_w, comp_w, "Women's: Standard vs Equal Category (top 20)")
plt.suptitle('Rank Changes Under Equal Category Share — 2025 NCAA Outdoor', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('rank_changes_equal_cat.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
def relay_dep_plot(ax, alt, std, label):
    t = (
        alt.merge(std, on='school')
        .assign(relay_pct=lambda x: x['pts_relay'] / x['pts_standard'] * 100)
        .query('pts_relay > 0')
        .sort_values('relay_pct', ascending=True)
        .tail(12)
    )
    ax.barh(t['school'], t['relay_pct'], color='#9467bd', alpha=0.85)
    ax.axvline(t['relay_pct'].mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
    ax.set_xlabel('Relay % of Total Score')
    ax.set_title(f"{label} — Relay Dependence", fontweight='bold')
    ax.legend(fontsize=8)

fig, (ax_m, ax_w) = plt.subplots(1, 2, figsize=(14, 6))
relay_dep_plot(ax_m, alt2ev_m, std_m, "Men's")
relay_dep_plot(ax_w, alt2ev_w, std_w, "Women's")
plt.suptitle('Relay Dependence — Most Exposed Under Strict Event Caps', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('relay_dependence.png', bbox_inches='tight', dpi=150)
plt.show()